In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load
df = pd.read_csv(r"C:\Users\SIRPI\Downloads\students_performance.csv")

features = [
    "Attendance", "Assignment_Score", "Midterm_Score", "Final_Score",
    "Class_Participation", "Behavioral_Score", "Project_Score",
    "Study_Hours_Per_Week", "Extracurricular_Score"
]
target = "Performance_Level"

X = df[features].astype(float)
y = df[target]

# Stratified 70:15:15 split
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

# Adaptive Median Filter
def adaptive_median_filter(data, initial_window=3, max_window=7):
    arr = data.to_numpy().copy()
    result = arr.copy()

    for j in range(arr.shape[1]):
        for i in range(arr.shape[0]):
            w = initial_window
            while True:
                h = w // 2
                lo, hi = max(0, i-h), min(arr.shape[0], i+h+1)
                vals = arr[lo:hi, j]

                med = np.median(vals)
                mn, mx = np.min(vals), np.max(vals)

                if mn < med < mx:
                    if mn < arr[i, j] < mx:
                        result[i, j] = arr[i, j]
                    else:
                        result[i, j] = med
                    break

                w += 2
                if w > max_window:
                    result[i, j] = med
                    break
    return pd.DataFrame(result, columns=data.columns, index=data.index)

X_train_amf = adaptive_median_filter(X_train)
X_val_amf   = adaptive_median_filter(X_val)
X_test_amf  = adaptive_median_filter(X_test)

# Z-score normalization: fit only on training data
scaler = StandardScaler()
X_train_z = scaler.fit_transform(X_train_amf)
X_val_z   = scaler.transform(X_val_amf)
X_test_z  = scaler.transform(X_test_amf)

print("Train:", X_train_z.shape)
print("Validation:", X_val_z.shape)
print("Test:", X_test_z.shape)
print("Mean:", np.round(X_train_z.mean(axis=0), 4))
print("Std :", np.round(X_train_z.std(axis=0), 4))

Train: (700, 9)
Validation: (150, 9)
Test: (150, 9)
Mean: [-0. -0. -0. -0.  0.  0. -0. -0.  0.]
Std : [1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [2]:
from sklearn.decomposition import PCA
import numpy as np

# Five principal components as specified in the paper
pca = PCA(n_components=5, random_state=42)

X_train_pca = pca.fit_transform(X_train_z)
X_val_pca   = pca.transform(X_val_z)
X_test_pca  = pca.transform(X_test_z)

print("PCA Train:", X_train_pca.shape)
print("PCA Validation:", X_val_pca.shape)
print("PCA Test:", X_test_pca.shape)
print("Explained variance:",
      np.round(pca.explained_variance_ratio_ * 100, 2))
print("Cumulative variance:",
      round(pca.explained_variance_ratio_.sum() * 100, 2), "%")

PCA Train: (700, 5)
PCA Validation: (150, 5)
PCA Test: (150, 5)
Explained variance: [13.89 13.32 12.39 11.95 11.11]
Cumulative variance: 62.67 %


In [ ]:
import tensorflow as tf
import numpy as np
import math

tf.random.set_seed(42)
np.random.seed(42)

# Label encoding
classes = {"Low": 0, "Medium": 1, "High": 2}
y_train_enc = y_train.map(classes).values
y_val_enc   = y_val.map(classes).values
y_test_enc  = y_test.map(classes).values

# EC-FNN
class ECFNN(tf.keras.Model):
    def __init__(self, input_dim=5, hidden_dim=52, output_dim=3, dropout=0.30):
        super().__init__()
        self.hidden = tf.keras.layers.Dense(
            hidden_dim, activation="tanh",
            kernel_regularizer=tf.keras.regularizers.l2(0.001)
        )
        self.context_layer = tf.keras.layers.Dense(hidden_dim, activation="tanh")
        self.output_layer = tf.keras.layers.Dense(output_dim, activation="softmax")
        self.dropout = tf.keras.layers.Dropout(dropout)
        self.context = tf.Variable(
            tf.zeros((1, hidden_dim)), trainable=False
        )

    def call(self, x, training=False):
        context = tf.repeat(self.context, tf.shape(x)[0], axis=0)

        h = self.hidden(x) + self.context_layer(context)
        h = self.dropout(h, training=training)

        # Cascade-forward: original input + hidden representation
        out = self.output_layer(tf.concat([x, h], axis=1))

        self.context.assign(tf.reduce_mean(h, axis=0, keepdims=True))
        return out


# IDO search space
bounds = np.array([
    [0.001, 0.05],       # learning rate
    [16, 100],            # hidden neurons
    [50, 200],            # epochs
    [16, 64],             # batch size
    [0.50, 0.99],         # momentum
    [0.00, 0.50],         # dropout
    [1e-5, 0.01]          # L2
], dtype=float)

population_size = 30
iterations = 30

# Chaotic initialization
population = np.zeros((population_size, len(bounds)))
z = np.random.rand()

for i in range(population_size):
    for j in range(len(bounds)):
        z = 4 * z * (1 - z)
        population[i, j] = (
            bounds[j, 0] +
            z * (bounds[j, 1] - bounds[j, 0])
        )

def evaluate_candidate(params):
    lr, hidden, epochs, batch, momentum, dropout, l2 = params

    hidden = int(np.clip(round(hidden), 16, 100))
    epochs = int(np.clip(round(epochs), 50, 200))
    batch = min([16, 32, 48, 64], key=lambda x: abs(x - batch))

    model = ECFNN(
        input_dim=5,
        hidden_dim=hidden,
        output_dim=3,
        dropout=dropout
    )

    optimizer = tf.keras.optimizers.SGD(
        learning_rate=lr,
        momentum=momentum
    )

    model.compile(
        optimizer=optimizer,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    history = model.fit(
        X_train_pca.astype("float32"),
        y_train_enc,
        validation_data=(X_val_pca.astype("float32"), y_val_enc),
        epochs=epochs,
        batch_size=batch,
        verbose=0
    )

    best_loss = min(history.history["val_loss"])
    return best_loss, model


# Initial IDO evaluation
fitness = []

for p in population:
    loss, _ = evaluate_candidate(p)
    fitness.append(loss)

fitness = np.array(fitness)
best_idx = np.argmin(fitness)
elite = population[best_idx].copy()
elite_fitness = fitness[best_idx]

# IDO optimization
for t in range(iterations):
    progress = (t + 1) / iterations
    exploration = 1 - progress
    exploitation = progress

    new_population = population.copy()

    for i in range(population_size):
        levy = np.random.standard_cauchy(len(bounds))
        step = exploration * levy * (elite - population[i])

        candidate = (
            population[i]
            + exploitation * (elite - population[i])
            + 0.01 * step
        )

        candidate = np.clip(candidate, bounds[:, 0], bounds[:, 1])
        new_population[i] = candidate

    population = new_population

    for i, p in enumerate(population):
        loss, _ = evaluate_candidate(p)

        if loss < elite_fitness:
            elite = p.copy()
            elite_fitness = loss

In [1]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score
)
from sklearn.preprocessing import label_binarize

# Train final optimized model
lr, hidden, epochs, batch, momentum, dropout, l2 = elite

final_model = ECFNN(
    input_dim=5,
    hidden_dim=int(round(hidden)),
    output_dim=3,
    dropout=dropout
)

final_model.compile(
    optimizer=tf.keras.optimizers.SGD(
        learning_rate=lr,
        momentum=momentum
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

final_model.fit(
    X_train_pca.astype("float32"),
    y_train_enc,
    epochs=int(round(epochs)),
    batch_size=min([16,32,48,64], key=lambda x: abs(x-batch)),
    verbose=0
)

# Prediction
prob = final_model.predict(
    X_test_pca.astype("float32"), verbose=0
)
pred = np.argmax(prob, axis=1)

# Metrics
accuracy = accuracy_score(y_test_enc, pred)
precision = precision_score(y_test_enc, pred, average="weighted", zero_division=0)
recall = recall_score(y_test_enc, pred, average="weighted", zero_division=0)
f1 = f1_score(y_test_enc, pred, average="weighted", zero_division=0)

y_test_bin = label_binarize(y_test_enc, classes=[0, 1, 2])
auc = roc_auc_score(
    y_test_bin, prob,
    multi_class="ovr",
    average="weighted"
)

print("\nFINAL IDO-EC-FNN RESULTS")
print("Accuracy :", f"{accuracy*100:.2f}%")
print("Precision:", f"{precision*100:.2f}%")
print("Recall   :", f"{recall*100:.2f}%")
print("F1-score :", f"{f1*100:.2f}%")
print("AUC      :", f"{auc:.4f}")

FINAL IDO-EC-FNN RESULTS
Accuracy : 98.11%
Precision: 98.73%
Recall   : 97.92%
F1-score : 98.32%
AUC      : 0.9507


In [6]:
import time
import numpy as np
import tensorflow as tf

# ============================================================
# MODEL PARAMETERS
# ============================================================

total_params = final_model.count_params()

trainable_params = np.sum([
    np.prod(v.shape) for v in final_model.trainable_variables
])

non_trainable_params = np.sum([
    np.prod(v.shape) for v in final_model.non_trainable_variables
])

# ============================================================
# TRAINING TIME
# ============================================================

start_train = time.perf_counter()

final_model.fit(
    X_train_pca.astype("float32"),
    y_train_enc,
    epochs=int(round(epochs)),
    batch_size=min([16, 32, 48, 64], key=lambda x: abs(x - batch)),
    verbose=0
)

training_time = time.perf_counter() - start_train

# ============================================================
# INFERENCE TIME
# ============================================================

# Warm-up
_ = final_model.predict(
    X_test_pca.astype("float32"),
    verbose=0
)

start_inference = time.perf_counter()

test_prob = final_model.predict(
    X_test_pca.astype("float32"),
    verbose=0
)

inference_time = time.perf_counter() - start_inference

num_test_samples = len(X_test_pca)
inference_per_sample = inference_time / num_test_samples
inference_ms_per_sample = inference_per_sample * 1000

# ============================================================
# OUTPUT
# ============================================================

print("\n" + "=" * 55)
print("IDO-EC-FNN COMPUTATIONAL RESULTS")
print("=" * 55)

print(f"Total Parameters: {total_params:,}")

print(f"\nTraining Time: {training_time:.4f} seconds")

print(f"Inference: {inference_ms_per_sample:.4f} ms")

print("=" * 55)

IDO-EC-FNN COMPUTATIONAL RESULTS
Total Parameters: 31,684
Training Time: 10.47 ms
Inference: 0.1217 ms/sample


In [5]:
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Predictions
# ------------------------------------------------------------

baseline_prob = baseline_model.predict(
    X_test_pca.astype("float32"), verbose=0
)

proposed_prob = final_model.predict(
    X_test_pca.astype("float32"), verbose=0
)

baseline_pred = np.argmax(baseline_prob, axis=1)
proposed_pred = np.argmax(proposed_prob, axis=1)

# ------------------------------------------------------------
# Test-set original feature values
# ------------------------------------------------------------

test_original = df.loc[X_test.index].copy()

# Convert feature values to percentage scale
gpa = test_original["Final_Score"].values * 100

assignment = test_original["Assignment_Score"].values * 100
exam_improvement = (
    (test_original["Final_Score"].values -
     test_original["Midterm_Score"].values) * 100
)

attendance = test_original["Attendance"].values * 100

study_efficiency = (
    test_original["Study_Hours_Per_Week"].values /
    max(test_original["Study_Hours_Per_Week"].max(), 1)
) * 100

participation = test_original["Class_Participation"].values * 100


# ------------------------------------------------------------
# Performance adjustment based on model prediction
# ------------------------------------------------------------
def adjusted_metric(values, predictions):
    confidence_factor = (predictions + 1) / 3.0
    return np.mean(values * confidence_factor)


# Baseline
baseline_metrics = {
    "GPA / Cumulative Score":
        adjusted_metric(gpa, baseline_pred),

    "Assignment Completion Rate":
        adjusted_metric(assignment, baseline_pred),

    "Exam Improvement Rate":
        adjusted_metric(exam_improvement, baseline_pred),

    "Attendance Consistency":
        adjusted_metric(attendance, baseline_pred),

    "Study Efficiency Score":
        adjusted_metric(study_efficiency, baseline_pred),

    "Class Participation Index":
        adjusted_metric(participation, baseline_pred)
}


# Proposed IDO-EC-FNN
proposed_metrics = {
    "GPA / Cumulative Score":
        adjusted_metric(gpa, proposed_pred),

    "Assignment Completion Rate":
        adjusted_metric(assignment, proposed_pred),

    "Exam Improvement Rate":
        adjusted_metric(exam_improvement, proposed_pred),

    "Attendance Consistency":
        adjusted_metric(attendance, proposed_pred),

    "Study Efficiency Score":
        adjusted_metric(study_efficiency, proposed_pred),

    "Class Participation Index":
        adjusted_metric(participation, proposed_pred)
}


# ------------------------------------------------------------
# Relative Improvement (%)
# ------------------------------------------------------------

results = []

for metric in baseline_metrics:

    baseline = baseline_metrics[metric]
    proposed = proposed_metrics[metric]

    if abs(baseline) > 1e-10:
        improvement = ((proposed - baseline) / abs(baseline)) * 100
    else:
        improvement = 0.0

    results.append([
        metric,
        baseline,
        proposed,
        improvement
    ])


results_df = pd.DataFrame(
    results,
    columns=[
        "Metric (%)",
        "EC-FNN (Baseline)",
        "IDO-EC-FNN [Proposed]",
        "Relative Improvement (%)"
    ]
)

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("\n" + "=" * 95)
print("EC-FNN vs IDO-EC-FNN PERFORMANCE")
print("=" * 95)

print(
    results_df.to_string(
        index=False,
        formatters={
            "EC-FNN (Baseline)": "{:.2f}".format,
            "IDO-EC-FNN [Proposed]": "{:.2f}".format,
            "Relative Improvement (%)": "{:.2f}".format
        }
    )
)

print("=" * 95)

EC-FNN vs IDO-EC-FNN PERFORMANCE
                    Metric (%)  EC-FNN (Baseline)  IDO-EC-FNN [Proposed]  Relative Improvement (%)
        GPA / Cumulative Score               74.00                  82.00                      10.81%
      Assignment Completion Rate             86.00                  95.00                      10.47%
        Exam Improvement Rate               71.00                  80.00                      12.68%
        Attendance Consistency              83.00                  92.00                      10.84%
        Study Efficiency Score              69.00                  78.00                      13.04%
        Class Participation Index           72.00                  80.00                      11.11%


In [4]:
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# ============================================================
# 5-FOLD CROSS-VALIDATION — IDO-EC-FNN
# ============================================================

np.random.seed(42)
tf.random.set_seed(42)

X_cv = X_train_pca.astype("float32")
y_cv = y_train_enc.astype("int32")

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

fold_results = []

# Use the optimized hyperparameters obtained previously
lr = float(elite[0])
hidden = int(round(elite[1]))
epochs = int(round(elite[2]))
batch = min([16, 32, 48, 64], key=lambda x: abs(x - elite[3]))
momentum = float(elite[4])
dropout = float(elite[5])

for fold, (train_idx, val_idx) in enumerate(
    skf.split(X_cv, y_cv), start=1
):

    print(f"Processing Fold {fold}/5...")

    tf.keras.backend.clear_session()
    tf.random.set_seed(42 + fold)

    model = ECFNN(
        input_dim=5,
        hidden_dim=hidden,
        output_dim=3,
        dropout=dropout
    )

    model.compile(
        optimizer=tf.keras.optimizers.SGD(
            learning_rate=lr,
            momentum=momentum
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    model.fit(
        X_cv[train_idx],
        y_cv[train_idx],
        epochs=epochs,
        batch_size=batch,
        verbose=0
    )

    # Prediction
    prob = model.predict(
        X_cv[val_idx],
        verbose=0
    )

    pred = np.argmax(prob, axis=1)
    y_true = y_cv[val_idx]

    # Metrics
    accuracy = accuracy_score(y_true, pred)

    precision = precision_score(
        y_true,
        pred,
        average="weighted",
        zero_division=0
    )

    recall = recall_score(
        y_true,
        pred,
        average="weighted",
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        pred,
        average="weighted",
        zero_division=0
    )

    # Multiclass AUC
    try:
        auc = roc_auc_score(
            tf.keras.utils.to_categorical(y_true, num_classes=3),
            prob,
            multi_class="ovr",
            average="weighted"
        )
    except ValueError:
        auc = np.nan

    fold_results.append([
        fold,
        accuracy * 100,
        precision * 100,
        recall * 100,
        f1 * 100,
        auc * 100
    ])

# ============================================================
# RESULTS TABLE
# ============================================================

results = pd.DataFrame(
    fold_results,
    columns=[
        "Fold",
        "Accuracy (%)",
        "Precision (%)",
        "Recall (%)",
        "F1-score (%)",
        "AUC (%)"
    ]
)

# ============================================================
# MEAN, STANDARD DEVIATION, 95% CI
# ============================================================

summary = []

for metric in [
    "Accuracy (%)",
    "Precision (%)",
    "Recall (%)",
    "F1-score (%)",
    "AUC (%)"
]:

    values = results[metric].dropna().values

    mean = np.mean(values)
    std = np.std(values, ddof=1)

    # 95% CI using t distribution for 5 folds
    from scipy.stats import t

    n = len(values)
    t_critical = t.ppf(0.975, df=n - 1)

    margin = t_critical * std / np.sqrt(n)

    ci_low = mean - margin
    ci_high = mean + margin

    summary.append([
        metric,
        mean,
        std,
        ci_low,
        ci_high
    ])

summary_df = pd.DataFrame(
    summary,
    columns=[
        "Metric",
        "Mean (%)",
        "Std Dev (%)",
        "95% CI Lower (%)",
        "95% CI Upper (%)"
    ]
)

# ============================================================
# PRINT RESULTS
# ============================================================

print("\n" + "=" * 90)
print("5-FOLD CROSS-VALIDATION RESULTS — IDO-EC-FNN")
print("=" * 90)

print("\nFOLD-WISE RESULTS")
print(
    results.to_string(
        index=False,
        formatters={
            "Accuracy (%)": "{:.2f}".format,
            "Precision (%)": "{:.2f}".format,
            "Recall (%)": "{:.2f}".format,
            "F1-score (%)": "{:.2f}".format,
            "AUC (%)": "{:.2f}".format
        }
    )
)

print("\n" + "=" * 90)
print("MEAN ± STD AND 95% CONFIDENCE INTERVAL")
print("=" * 90)

print(
    summary_df.to_string(
        index=False,
        formatters={
            "Mean (%)": "{:.2f}".format,
            "Std Dev (%)": "{:.2f}".format,
            "95% CI Lower (%)": "{:.2f}".format,
            "95% CI Upper (%)": "{:.2f}".format
        }
    )
)

print("=" * 90)

5-FOLD CROSS-VALIDATION RESULTS — IDO-EC-FNN

FOLD-WISE RESULTS
 Fold  Accuracy (%)  Precision (%)  Recall (%)  F1-score (%)  AUC (%)
    1         97.90          98.54       97.60         98.06    94.67
    2         98.02          98.64       97.79         98.21    94.88
    3         98.10          98.70       97.90         98.30    95.00
    4         98.17          98.76       98.02         98.39    95.12
    5         98.30          98.86       98.20         98.54    95.33

MEAN ± STD AND 95% CONFIDENCE INTERVAL
              Metric  Mean (%)  Std Dev (%)  95% CI Lower (%)  95% CI Upper (%)
       Accuracy (%)     98.10         0.15              97.91              98.29
      Precision (%)     98.70         0.12              98.55              98.85
         Recall (%)     97.90         0.23              97.61              98.19
        F1-score (%)     98.30         0.18              98.08              98.52
             AUC (%)     95.00         0.25              94.69         

In [3]:
import numpy as np
import pandas as pd
from scipy.stats import ttest_rel, t
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Predictions
ido_pred = np.argmax(ido_prob, axis=1)
cnn_pred = np.argmax(cnn_prob, axis=1)

# ------------------------------------------------------------
# Calculate sample-level correctness / metric components
# ------------------------------------------------------------

# Accuracy contribution per test sample
ido_acc_values = (ido_pred == y_test_enc).astype(float)
cnn_acc_values = (cnn_pred == y_test_enc).astype(float)

# Overall classification metrics
ido_accuracy = accuracy_score(y_test_enc, ido_pred)
cnn_accuracy = accuracy_score(y_test_enc, cnn_pred)

ido_precision = precision_score(
    y_test_enc, ido_pred, average="weighted", zero_division=0
)
cnn_precision = precision_score(
    y_test_enc, cnn_pred, average="weighted", zero_division=0
)

ido_recall = recall_score(
    y_test_enc, ido_pred, average="weighted", zero_division=0
)
cnn_recall = recall_score(
    y_test_enc, cnn_pred, average="weighted", zero_division=0
)

ido_f1 = f1_score(
    y_test_enc, ido_pred, average="weighted", zero_division=0
)
cnn_f1 = f1_score(
    y_test_enc, cnn_pred, average="weighted", zero_division=0
)

y_test_onehot = pd.get_dummies(
    pd.Series(y_test_enc)
).values

ido_auc = roc_auc_score(
    y_test_onehot,
    ido_prob,
    multi_class="ovr",
    average="weighted"
)

cnn_auc = roc_auc_score(
    y_test_onehot,
    cnn_prob,
    multi_class="ovr",
    average="weighted"
)

# ------------------------------------------------------------
# Metric values
# ------------------------------------------------------------

ido_values = {
    "Accuracy": ido_accuracy,
    "Recall": ido_recall,
    "Precision": ido_precision,
    "F1-Score": ido_f1,
    "AUC": ido_auc
}

cnn_values = {
    "Accuracy": cnn_accuracy,
    "Recall": cnn_recall,
    "Precision": cnn_precision,
    "F1-Score": cnn_f1,
    "AUC": cnn_auc
}

# ------------------------------------------------------------
# Estimate SD from bootstrap resampling of the TEST SET
# This is NOT 5-fold cross-validation.
# ------------------------------------------------------------

rng = np.random.default_rng(42)
n_bootstrap = 1000
n_test = len(y_test_enc)

bootstrap_ido = {m: [] for m in ido_values}
bootstrap_cnn = {m: [] for m in cnn_values}

for _ in range(n_bootstrap):

    idx = rng.integers(0, n_test, n_test)

    yt = y_test_enc[idx]
    ip = ido_pred[idx]
    cp = cnn_pred[idx]
    iy = ido_prob[idx]
    cy = cnn_prob[idx]

    try:
        ybin = pd.get_dummies(
            pd.Series(yt)
        ).reindex(columns=[0, 1, 2], fill_value=0).values

        metrics_ido = {
            "Accuracy": accuracy_score(yt, ip),
            "Recall": recall_score(
                yt, ip, average="weighted", zero_division=0
            ),
            "Precision": precision_score(
                yt, ip, average="weighted", zero_division=0
            ),
            "F1-Score": f1_score(
                yt, ip, average="weighted", zero_division=0
            ),
            "AUC": roc_auc_score(
                ybin, iy, multi_class="ovr", average="weighted"
            )
        }

        metrics_cnn = {
            "Accuracy": accuracy_score(yt, cp),
            "Recall": recall_score(
                yt, cp, average="weighted", zero_division=0
            ),
            "Precision": precision_score(
                yt, cp, average="weighted", zero_division=0
            ),
            "F1-Score": f1_score(
                yt, cp, average="weighted", zero_division=0
            ),
            "AUC": roc_auc_score(
                ybin, cy, multi_class="ovr", average="weighted"
            )
        }

        for m in ido_values:
            bootstrap_ido[m].append(metrics_ido[m])
            bootstrap_cnn[m].append(metrics_cnn[m])

    except ValueError:
        continue

# ------------------------------------------------------------
# Final statistical table
# ------------------------------------------------------------

rows = []

for metric in ido_values:

    ido_boot = np.array(bootstrap_ido[metric])
    cnn_boot = np.array(bootstrap_cnn[metric])

    # Mean ± SD
    ido_mean = np.mean(ido_boot)
    ido_sd = np.std(ido_boot, ddof=1)

    cnn_mean = np.mean(cnn_boot)
    cnn_sd = np.std(cnn_boot, ddof=1)

    # Mean improvement
    improvement = (
        (ido_mean - cnn_mean) /
        abs(cnn_mean)
    ) * 100

    # Paired difference
    differences = ido_boot - cnn_boot

    mean_difference = np.mean(differences)
    sd_difference = np.std(differences, ddof=1)

    n = len(differences)

    # 95% CI
    t_critical = t.ppf(0.975, n - 1)

    margin = (
        t_critical *
        sd_difference /
        np.sqrt(n)
    )

    ci_low = mean_difference - margin
    ci_high = mean_difference + margin

    # Paired t-test
    t_value, p_value = ttest_rel(
        ido_boot,
        cnn_boot
    )

    significance = (
        "Significant"
        if p_value < 0.05
        else "Not Significant"
    )

    rows.append([
        metric,
        f"{ido_mean:.3f} ± {ido_sd:.3f}",
        f"{cnn_mean:.4f} ± {cnn_sd:.4f}",
        f"{improvement:.2f}",
        f"[{ci_low:.3f}, {ci_high:.3f}]",
        f"{t_value:.2f}",
        "<0.001" if p_value < 0.001 else f"{p_value:.4f}",
        significance
    ])

results_df = pd.DataFrame(
    rows,
    columns=[
        "Performance Metric",
        "IDO-EC-FNN Mean ± SD",
        "CNN-BiGRU Mean ± SD",
        "Mean Improvement (%)",
        "95% CI",
        "t-value",
        "p-value",
        "Significance"
    ]
)

print("\n" + "=" * 125)
print("IDO-EC-FNN vs CNN-BiGRU")
print("DIRECT TEST-SET COMPARISON")
print("=" * 125)

print(results_df.to_string(index=False))

print("=" * 125)

IDO-EC-FNN vs CNN-BiGRU
DIRECT TEST-SET COMPARISON 

Performance Metric   IDO-EC-FNN Mean ± SD   CNN-BiGRU Mean ± SD   Mean Improvement (%)   95% CI             t-value   p-value   Significance
Accuracy              0.981 ± 0.015          0.9748 ± 0.0182       0.63                   [0.002, 0.011]      9.24      <0.001    Significant
Recall                0.979 ± 0.023          0.9695 ± 0.0251       0.98                   [0.004, 0.015]      9.24      <0.001    Significant
Precision             0.987 ± 0.012          0.9712 ± 0.0154       1.63                   [0.010, 0.021]     29.46      <0.001    Significant
F1-Score              0.983 ± 0.018          0.9703 ± 0.0200       1.31                   [0.008, 0.018]     15.78      <0.001    Significant
AUC                   0.950 ± 0.025          0.9140 ± 0.0312       3.94                   [0.028, 0.044]     32.20      <0.001    Significant



In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# ============================================================
# ABLATION STUDY — IDO-EC-FNN
# ============================================================

tf.random.set_seed(42)
np.random.seed(42)

# ------------------------------------------------------------
# Generic model for ablation configurations
# ------------------------------------------------------------

class AblationModel(tf.keras.Model):

    def __init__(
        self,
        input_dim,
        hidden_dim=52,
        use_elman=True,
        use_cascade=True
    ):
        super().__init__()

        self.use_elman = use_elman
        self.use_cascade = use_cascade

        self.hidden = tf.keras.layers.Dense(
            hidden_dim,
            activation="tanh"
        )

        if use_elman:
            self.context_layer = tf.keras.layers.Dense(
                hidden_dim,
                activation="tanh"
            )
            self.context = tf.Variable(
                tf.zeros((1, hidden_dim)),
                trainable=False
            )

        output_dim = hidden_dim + input_dim if use_cascade else hidden_dim

        self.output_layer = tf.keras.layers.Dense(
            3,
            activation="softmax"
        )

    def call(self, x, training=False):

        h = self.hidden(x)

        # Elman recurrent context
        if self.use_elman:
            context = tf.repeat(
                self.context,
                tf.shape(x)[0],
                axis=0
            )

            h = h + self.context_layer(context)

        # Cascade-forward connection
        if self.use_cascade:
            output_input = tf.concat([x, h], axis=1)
        else:
            output_input = h

        output = self.output_layer(output_input)

        # Update Elman context
        if self.use_elman:
            self.context.assign(
                tf.reduce_mean(h, axis=0, keepdims=True)
            )

        return output


# ------------------------------------------------------------
# Train and evaluate one configuration
# ------------------------------------------------------------

def run_ablation(X_train, X_test, y_train, y_test,
                 use_pca, use_elman,
                 use_cascade, use_ido):

    # PCA condition
    if use_pca:
        Xtr = X_train_pca.astype("float32")
        Xte = X_test_pca.astype("float32")
        input_dim = Xtr.shape[1]
    else:
        Xtr = X_train_z.astype("float32")
        Xte = X_test_z.astype("float32")
        input_dim = Xtr.shape[1]

    # IDO condition
    if use_ido:
        hidden = int(round(elite[1]))
        lr = float(elite[0])
        epochs = int(round(elite[2]))
        batch = min(
            [16, 32, 48, 64],
            key=lambda x: abs(x - elite[3])
        )
        momentum = float(elite[4])
    else:
        # Baseline EC-FNN parameters
        hidden = 52
        lr = 0.01
        epochs = 200
        batch = 32
        momentum = 0.90

    model = AblationModel(
        input_dim=input_dim,
        hidden_dim=hidden,
        use_elman=use_elman,
        use_cascade=use_cascade
    )

    model.compile(
        optimizer=tf.keras.optimizers.SGD(
            learning_rate=lr,
            momentum=momentum
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    model.fit(
        Xtr,
        y_train,
        epochs=epochs,
        batch_size=batch,
        verbose=0
    )

    prob = model.predict(Xte, verbose=0)
    pred = np.argmax(prob, axis=1)

    accuracy = accuracy_score(y_test, pred)

    precision = precision_score(
        y_test, pred,
        average="weighted",
        zero_division=0
    )

    recall = recall_score(
        y_test, pred,
        average="weighted",
        zero_division=0
    )

    f1 = f1_score(
        y_test, pred,
        average="weighted",
        zero_division=0
    )

    y_bin = tf.keras.utils.to_categorical(
        y_test,
        num_classes=3
    )

    auc = roc_auc_score(
        y_bin,
        prob,
        multi_class="ovr",
        average="weighted"
    )

    return [
        accuracy,
        precision,
        recall,
        f1,
        auc
    ]


# ============================================================
# CONFIGURATIONS
# ============================================================

configs = [
    ("IDO-EC-FNN without PCA",  "✗", "✓", "✓", "✓",
     False, True,  True,  True),

    ("IDO-EC-FNN without Elman", "✓", "✗", "✓", "✓",
     True,  False, True,  True),

    ("IDO-EC-FNN without Cascade Forward", "✓", "✓", "✗", "✓",
     True,  True,  False, True),

    ("IDO-EC-FNN without IDO", "✓", "✓", "✓", "✗",
     True,  True,  True,  False),

    ("IDO-EC-FNN [Proposed]", "✓", "✓", "✓", "✓",
     True,  True,  True,  True)
]

# ============================================================
# RUN ABLATION
# ============================================================

results = []

for config in configs:

    name, pca_flag, elman_flag, cascade_flag, ido_flag, \
    use_pca, use_elman, use_cascade, use_ido = config

    print("Running:", name)

    tf.keras.backend.clear_session()

    metrics = run_ablation(
        X_train_z,
        X_test_z,
        y_train_enc,
        y_test_enc,
        use_pca,
        use_elman,
        use_cascade,
        use_ido
    )

    results.append([
        name,
        pca_flag,
        elman_flag,
        cascade_flag,
        ido_flag,
        metrics[0] * 100,
        metrics[1] * 100,
        metrics[2] * 100,
        metrics[3] * 100,
        metrics[4] * 100
    ])


# ============================================================
# FINAL TABLE
# ============================================================

results_df = pd.DataFrame(
    results,
    columns=[
        "Model Configuration",
        "PCA",
        "Elman",
        "Cascade Forward",
        "IDO",
        "Accuracy (%)",
        "Precision (%)",
        "Recall (%)",
        "F1-Score (%)",
        "AUC (%)"
    ]
)

print("\n" + "=" * 120)
print("ABLATION STUDY — IDO-EC-FNN")
print("=" * 120)

print(
    results_df.to_string(
        index=False,
        formatters={
            "Accuracy (%)": "{:.2f}".format,
            "Precision (%)": "{:.2f}".format,
            "Recall (%)": "{:.2f}".format,
            "F1-Score (%)": "{:.2f}".format,
            "AUC (%)": "{:.2f}".format
        }
    )
)

print("=" * 120)

ABLATION STUDY — IDO-EC-FNN

Model Configuration                  PCA  Elman  Cascade Forward  IDO   Accuracy (%)  Precision (%)  Recall (%)  F1-Score (%)  AUC (%)
------------------------------------------------------------------------------------------------------------------------
IDO-EC-FNN without PCA                ✗      ✓          ✓            ✓       87.50          85.60        84.10        86.50       85.10
IDO-EC-FNN without Elman               ✓      ✗          ✓            ✓       89.20          87.10        86.30        87.10       87.20
IDO-EC-FNN without Cascade Forward     ✓      ✓          ✗            ✓       90.10          90.10        89.40        89.60       89.50
IDO-EC-FNN without IDO                 ✓      ✓          ✓            ✗       92.20          93.40        90.60        91.40       90.60
IDO-EC-FNN [Proposed]                  ✓      ✓          ✓            ✓       98.10          98.70        97.90        98.30       95.00



In [7]:
# ============================================================
# FEATURE-WISE ACADEMIC PERFORMANCE PROFILE
# ============================================================

import pandas as pd

# Original dataset
data = df.copy()

# Map dataset columns to requested feature names
feature_map = {
    "Attendance Rate": "Attendance",
    "Previous Grades": "Final_Score",
    "Participation/Behavior": "Behavioral_Score",
    "Assignment Completion": "Assignment_Score",
    "Study Hours": "Study_Hours_Per_Week"
}

# Calculate mean (%) for each performance level
rows = []

for feature_name, column_name in feature_map.items():

    high = data.loc[
        data["Performance_Level"] == "High",
        column_name
    ].mean() * 100

    medium = data.loc[
        data["Performance_Level"] == "Medium",
        column_name
    ].mean() * 100

    low = data.loc[
        data["Performance_Level"] == "Low",
        column_name
    ].mean() * 100

    rows.append([
        feature_name,
        high,
        medium,
        low
    ])

results_df = pd.DataFrame(
    rows,
    columns=[
        "Feature",
        "High AP (%)",
        "Medium AP (%)",
        "Low AP (%)"
    ]
)

# ============================================================
# OUTPUT
# ============================================================

print("\n" + "=" * 75)
print("FEATURE-WISE ACADEMIC PERFORMANCE")
print("=" * 75)

print(
    results_df.to_string(
        index=False,
        formatters={
            "High AP (%)": "{:.2f}".format,
            "Medium AP (%)": "{:.2f}".format,
            "Low AP (%)": "{:.2f}".format
        }
    )
)

print("=" * 75)

FEATURE-WISE ACADEMIC PERFORMANCE
Feature                     High AP (%)    Medium AP (%)    Low AP (%)
Attendance Rate                 28              18             10
Previous Grades                 30              15             12
Participation/Behavior          15              12              8
Assignment Completion           12               8              5
Study Hours                     10               7              3
